#### Expected Size

In [19]:
# import os
# import cv2
# from tqdm import tqdm
# from collections import Counter

# # ============================================================
# # 🗂 PATHS TO ORIGINAL OKUTAMA FRAMES
# # ============================================================
# root_paths = [
#     r"okutama-action/TrainSetFrames",
#     r"okutama-action/TestSetFrames"
# ]

# expected_size = (3840, 2160)  # (width, height)
# size_counter = Counter()
# bad_images = []

# # ============================================================
# # 🔍 SCAN ALL EXTRACTED FRAMES
# # ============================================================
# for root in root_paths:
#     if not os.path.exists(root):
#         continue
#     print(f"\n🔍 Checking frame dimensions under: {root}")
#     for drone in sorted(os.listdir(root)):
#         drone_path = os.path.join(root, drone)
#         if not os.path.isdir(drone_path):
#             continue
#         for time_dir in sorted(os.listdir(drone_path)):
#             time_path = os.path.join(drone_path, time_dir)
#             extracted_path = os.path.join(time_path, "Extracted-Frames-1280x720")
#             if not os.path.isdir(extracted_path):
#                 continue
#             for video_folder in sorted(os.listdir(extracted_path)):
#                 video_path = os.path.join(extracted_path, video_folder)
#                 if not os.path.isdir(video_path):
#                     continue
#                 frame_files = [f for f in os.listdir(video_path) if f.lower().endswith((".jpg", ".png"))]
#                 for frame_name in tqdm(frame_files, desc=f"{video_folder}", leave=False):
#                     frame_path = os.path.join(video_path, frame_name)
#                     img = cv2.imread(frame_path)
#                     if img is None:
#                         bad_images.append(frame_path)
#                         continue
#                     h, w = img.shape[:2]
#                     size_counter[(w, h)] += 1

# # ============================================================
# # 📊 SUMMARY
# # ============================================================
# print("\n======================================================")
# print("📏 Frame Resolution Summary")
# print("======================================================")
# for (w, h), count in size_counter.most_common():
#     mark = "✅" if (w, h) == expected_size else "⚠️"
#     print(f"{mark} {w}×{h} → {count} frames")

# if bad_images:
#     print("\n⚠️ Failed to read the following images (showing 5):")
#     for b in bad_images[:5]:
#         print("   ", b)
# else:
#     print("\n✅ No corrupted frames detected.")

# # ============================================================
# # 🧠 RESULT INTERPRETATION
# # ============================================================
# if len(size_counter) == 1 and next(iter(size_counter.keys())) == expected_size:
#     print(f"\n🎯 All extracted frames are consistently {expected_size[0]}×{expected_size[1]} ✅")
# else:
#     print("\n⚠️ Inconsistent frame sizes found! You might have mixed resolutions.")


In [20]:
# import os
# import cv2
# from tqdm import tqdm

# # ============================================================
# # PATHS
# # ============================================================
# frames_root = "okutama-action/TrainSetFrames"
# labels_dir  = "okutama-action/TrainSetFrames/Labels/SingleActionLabels/3840x2160"
# output_root = "dataset-yolo-person"

# # ============================================================
# # PARAMETERS
# # ============================================================
# ORIG_W, ORIG_H = 3840, 2160
# CLASS_ID = 0
# SPLIT = "train"

# # ============================================================
# # OUTPUT FOLDERS
# # ============================================================
# img_out_dir = os.path.join(output_root, "images", SPLIT)
# lbl_out_dir = os.path.join(output_root, "labels", SPLIT)

# os.makedirs(img_out_dir, exist_ok=True)
# os.makedirs(lbl_out_dir, exist_ok=True)

# # ============================================================
# # HELPER
# # ============================================================
# def convert_to_yolo(x1, y1, x2, y2, img_w, img_h):
#     bw = x2 - x1
#     bh = y2 - y1
#     cx = x1 + bw / 2
#     cy = y1 + bh / 2
#     return cx / img_w, cy / img_h, bw / img_w, bh / img_h

# # ============================================================
# # MAIN LOOP
# # ============================================================
# sample_idx = 0

# for drone in sorted(os.listdir(frames_root)):
#     drone_path = os.path.join(frames_root, drone)
#     if not os.path.isdir(drone_path):
#         continue

#     for time_dir in sorted(os.listdir(drone_path)):
#         time_path = os.path.join(drone_path, time_dir)
#         extracted_path = os.path.join(time_path, "Extracted-Frames-1280x720")

#         if not os.path.isdir(extracted_path):
#             continue

#         for video_folder in sorted(os.listdir(extracted_path)):
#             video_path = os.path.join(extracted_path, video_folder)
#             if not os.path.isdir(video_path):
#                 continue

#             label_file = os.path.join(labels_dir, f"{video_folder}.txt")
#             if not os.path.exists(label_file):
#                 continue

#             # ====================================================
#             # BUILD FRAME MAP
#             # ====================================================
#             frame_map = {}

#             with open(label_file, "r") as f:
#                 for line in f:
#                     parts = line.strip().split()
#                     if len(parts) < 10:
#                         continue

#                     try:
#                         frame_id   = int(parts[5])
#                         lost       = int(parts[6])
#                         label_name = parts[9].strip('"')
#                     except:
#                         continue

#                     if label_name.lower() != "person":
#                         continue
#                     if lost == 1:
#                         continue

#                     # ✅ allow generated (important)
#                     frame_map.setdefault(frame_id, []).append(parts)

#             # ====================================================
#             # CORRECT NUMERIC SORT
#             # ====================================================
#             frame_files = sorted(
#                 [f for f in os.listdir(video_path) if f.endswith(".jpg")],
#                 key=lambda x: int(os.path.splitext(x)[0])
#             )

#             # ====================================================
#             # LOOP FRAMES
#             # ====================================================
#             for f in tqdm(frame_files, desc=video_folder):

#                 frame_num = int(os.path.splitext(f)[0])  # 🔥 FIX

#                 img_path = os.path.join(video_path, f)
#                 img = cv2.imread(img_path)
#                 if img is None:
#                     continue

#                 img_h, img_w = img.shape[:2]
#                 yolo_lines = []

#                 # ====================================================
#                 # MATCH USING frame_num
#                 # ====================================================
#                 for parts in frame_map.get(frame_num, []):

#                     x1 = float(parts[1]) * (img_w / ORIG_W)
#                     y1 = float(parts[2]) * (img_h / ORIG_H)
#                     x2 = float(parts[3]) * (img_w / ORIG_W)
#                     y2 = float(parts[4]) * (img_h / ORIG_H)

#                     # clip
#                     x1 = max(0, min(x1, img_w))
#                     y1 = max(0, min(y1, img_h))
#                     x2 = max(0, min(x2, img_w))
#                     y2 = max(0, min(y2, img_h))

#                     if x2 <= x1 or y2 <= y1:
#                         continue

#                     cx, cy, bw, bh = convert_to_yolo(x1, y1, x2, y2, img_w, img_h)

#                     if bw <= 0 or bh <= 0:
#                         continue

#                     yolo_lines.append(
#                         f"{CLASS_ID} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f}"
#                     )

#                 # ====================================================
#                 # SKIP EMPTY (important for training)
#                 # ====================================================
#                 if len(yolo_lines) == 0:
#                     continue

#                 # ====================================================
#                 # CORRECT NAMING
#                 # ====================================================
#                 out_name = f"{drone}_{video_folder}_{frame_num:06d}.jpg"

#                 out_img_path = os.path.join(img_out_dir, out_name)
#                 out_lbl_path = os.path.join(lbl_out_dir, out_name.replace(".jpg", ".txt"))

#                 cv2.imwrite(out_img_path, img)

#                 with open(out_lbl_path, "w") as f:
#                     f.write("\n".join(yolo_lines))

#                 sample_idx += 1

#                 if sample_idx % 2000 == 0:
#                     print(f"[INFO] Processed {sample_idx:,} samples...")

# print("\n✅ TRAIN conversion completed!")
# print(f"Total samples saved: {sample_idx}")

<!-- Debug code -->